In [ ]:
import pandas as pd
import os

PASTA_IMAGENS = 'images'

df_treino = pd.read_csv('split_treino.csv')
df_val    = pd.read_csv('split_val.csv')
df_teste  = pd.read_csv('split_teste.csv')

print("treino:", len(df_treino), "| val:", len(df_val), "| teste:", len(df_teste))

In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import cv2
import os

IMG_SIZE = 224
MEDIA = [0.485, 0.456, 0.406]
DESVIO = [0.229, 0.224, 0.225]
BATCH = 32

class CLAHE:
    def __call__(self, img):
        arr = np.array(img)
        lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        arr = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        return Image.fromarray(arr)

transform_treino = transforms.Compose([
    CLAHE(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA, DESVIO),
])
transform_avaliacao = transforms.Compose([
    CLAHE(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA, DESVIO),
])

class DatasetRetina(Dataset):
    def __init__(self, dataframe, pasta_imagens, transform):
        self.df = dataframe.reset_index(drop=True)
        self.pasta = pasta_imagens
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        linha = self.df.iloc[i]
        caminho = os.path.join(self.pasta, linha['file'])
        imagem = Image.open(caminho).convert('RGB')
        imagem = self.transform(imagem)
        grau = int(linha['final_icdr'])
        return imagem, grau

ds_treino = DatasetRetina(df_treino, PASTA_IMAGENS, transform_treino)
ds_val    = DatasetRetina(df_val,    PASTA_IMAGENS, transform_avaliacao)
ds_teste  = DatasetRetina(df_teste,  PASTA_IMAGENS, transform_avaliacao)

dl_treino = DataLoader(ds_treino, batch_size=BATCH, shuffle=True,  num_workers=0)
dl_val    = DataLoader(ds_val,    batch_size=BATCH, shuffle=False, num_workers=0)
dl_teste  = DataLoader(ds_teste,  batch_size=BATCH, shuffle=False, num_workers=0)
print("treino:", len(dl_treino), "| val:", len(dl_val), "| teste:", len(dl_teste))

In [ ]:
import timm, torch
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

device = 'mps' if torch.backends.mps.is_available() else 'cpu'

modelo = timm.create_model('resnet18', pretrained=True, num_classes=5)
modelo = modelo.to(device)

classes = np.array([0, 1, 2, 3, 4])
pesos = compute_class_weight('balanced', classes=classes, y=df_treino['final_icdr'].values)
pesos_tensor = torch.tensor(pesos, dtype=torch.float32).to(device)

criterio = torch.nn.CrossEntropyLoss(weight=pesos_tensor)
otimizador = torch.optim.AdamW(modelo.parameters(), lr=1e-4)
print("pronto")

In [ ]:
from sklearn.metrics import cohen_kappa_score, accuracy_score

def avaliar(loader):
    modelo.eval()
    preds_todos, reais_todos, perda_total = [], [], 0
    with torch.no_grad():
        for imagens, graus in loader:
            imagens, graus = imagens.to(device), graus.to(device)
            saida = modelo(imagens)
            perda_total += criterio(saida, graus).item()
            preds_todos.extend(saida.argmax(dim=1).cpu().numpy())
            reais_todos.extend(graus.cpu().numpy())
    qwk = cohen_kappa_score(reais_todos, preds_todos, weights='quadratic')
    acc = accuracy_score(reais_todos, preds_todos)
    return perda_total/len(loader), qwk, acc

EPOCAS = 15
melhor_qwk = -1
CAMINHO_MODELO = 'melhor_modelo_resnet18_clahe.pth'

for epoca in range(1, EPOCAS + 1):
    modelo.train()
    perda_treino = 0
    for imagens, graus in dl_treino:
        imagens, graus = imagens.to(device), graus.to(device)
        otimizador.zero_grad()
        saida = modelo(imagens)
        perda = criterio(saida, graus)
        perda.backward()
        otimizador.step()
        perda_treino += perda.item()
    perda_treino /= len(dl_treino)

    perda_val, qwk_val, acc_val = avaliar(dl_val)
    print(f"época {epoca:2d} | perda treino {perda_treino:.3f} | perda val {perda_val:.3f} | QWK val {qwk_val:.3f} | acurácia {acc_val:.3f}")

    if qwk_val > melhor_qwk:
        melhor_qwk = qwk_val
        torch.save(modelo.state_dict(), CAMINHO_MODELO)

print(f"\ntreino concluído, melhor QWK de validação: {melhor_qwk:.3f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score
import matplotlib.pyplot as plt
import numpy as np
import torch

modelo.load_state_dict(torch.load('melhor_modelo_resnet18_clahe.pth'))
modelo.eval()

preds, reais = [], []
with torch.no_grad():
    for imagens, graus in dl_teste:
        imagens = imagens.to(device)
        saida = modelo(imagens)
        preds.extend(saida.argmax(dim=1).cpu().numpy())
        reais.extend(graus.numpy())

preds, reais = np.array(preds), np.array(reais)

qwk_teste = cohen_kappa_score(reais, preds, weights='quadratic')

nomes = ['0-Sem RD', '1-Leve', '2-Moderada', '3-Grave', '4-Proliferativa']
print(classification_report(reais, preds, target_names=nomes, digits=3, zero_division=0))

cm = confusion_matrix(reais, preds)
plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(5), nomes, rotation=45, ha='right')
plt.yticks(range(5), nomes)
plt.xlabel('Previsto pelo modelo')
plt.ylabel('Grau verdadeiro')
plt.title(f'ResNet18 + CLAHE (Teste) — QWK {qwk_teste:.3f}')
for i in range(5):
    for j in range(5):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.tight_layout()
plt.show()

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import torch, os

modelo.load_state_dict(torch.load('melhor_modelo_resnet18_clahe.pth'))
modelo.eval()

cam = GradCAM(model=modelo, target_layers=[modelo.layer4[-1]])

def desnormalizar(t):
    img = t.numpy().transpose(1, 2, 0)
    img = img * np.array(DESVIO) + np.array(MEDIA)
    return np.clip(img, 0, 1).astype(np.float32)

graus_alvo = [4, 4, 3, 2, 0]
selecionadas, cont = [], {}
for g in graus_alvo:
    linhas = df_teste[df_teste['final_icdr'] == g].reset_index(drop=True)
    c = cont.get(g, 0)
    selecionadas.append(linhas.iloc[c])
    cont[g] = c + 1

n = len(selecionadas)
plt.figure(figsize=(13, 5))
for k, linha in enumerate(selecionadas):
    caminho = os.path.join(PASTA_IMAGENS, linha['file'])
    entrada = transform_avaliacao(Image.open(caminho).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        previsto = modelo(entrada).argmax(dim=1).item()
    real = int(linha['final_icdr'])

    mapa = cam(input_tensor=entrada, targets=[ClassifierOutputTarget(previsto)])[0]
    base = desnormalizar(entrada[0].cpu())
    sobreposto = show_cam_on_image(base, mapa, use_rgb=True)

    plt.subplot(2, n, k + 1); plt.imshow(base); plt.axis('off')
    plt.title(f"Real: {real} | Previu: {previsto}", fontsize=10)
    plt.subplot(2, n, n + k + 1); plt.imshow(sobreposto); plt.axis('off')
plt.tight_layout(); plt.show()